# VL Attention Extraction — 4080 32GB version

This notebook is the cell-by-cell version of `extract_attention.py`. Run it top to bottom.

**期望产出**：`./vl_attention_data.json`，可直接拷贝到 `src/data/`。

**对比 Colab 版本的修复**：
1. CLIP 用一组**固定的 VQA 高频候选答案**评分，避免泄漏
2. CLIP 取 CLS-token 对 patch 的注意力（不是整个 49×49 自相关），并归一化到 \[0,1\]
3. BLIP2 cross-attention 通过 hook 捕获 4D `(B,heads,Q,K)` 张量，正确切片成 14×14 patch grid
4. BLIP2 推理结果**剥掉 prompt 回声**再判分
5. 只下载 VQA JSON（约 200MB），图片**按需下载**——不再下 6.2GB val2014.zip
6. 多加一个 BLIP2-FLAN-T5-XL 作第三个模型
7. 多输出一个 `text_bias`（视图五）数据

In [ ]:
# 一次安装。autodl 的镜像里已经有 torch 2.7+cu128，所以这里不动它。
!pip install -q transformers accelerate sentencepiece open_clip_torch pillow tqdm

In [ ]:
import os, json, gc, random, functools, urllib.request, zipfile
from pathlib import Path
from io import BytesIO
# autodl 内地机器需要走 HF 镜像
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '0')
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if device == 'cuda' else torch.float32
print('device', device, DTYPE)
if device == 'cuda':
    print('gpu', torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

DATA_DIR = Path('./vqa_data'); DATA_DIR.mkdir(exist_ok=True, parents=True)
IMG_DIR  = DATA_DIR / 'val2014'; IMG_DIR.mkdir(exist_ok=True, parents=True)
NUM_SAMPLES = 200            # 在 4080 上 ~10 分钟跑完全部 3 个模型
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
Q_URL = 'https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip'
A_URL = 'https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip'

def _dl_unzip(url, dst):
    z = dst / 'tmp.zip'
    print('downloading', url)
    urllib.request.urlretrieve(url, z)
    with zipfile.ZipFile(z) as f: f.extractall(dst)
    z.unlink()

q_json = DATA_DIR / 'v2_OpenEnded_mscoco_val2014_questions.json'
a_json = DATA_DIR / 'v2_mscoco_val2014_annotations.json'
if not q_json.exists(): _dl_unzip(Q_URL, DATA_DIR)
if not a_json.exists(): _dl_unzip(A_URL, DATA_DIR)

questions = json.load(open(q_json))['questions']
annos = {a['question_id']: a for a in json.load(open(a_json))['annotations']}
print('total questions', len(questions))

In [ ]:
# 按问题前缀分桶采样，避免 yes/no 主导
buckets = {'what':[], 'how':[], 'where':[], 'is_are':[], 'other':[]}
for q in questions:
    t = q['question'].lower()
    if   t.startswith('what'):  buckets['what'].append(q)
    elif t.startswith('how'):   buckets['how'].append(q)
    elif t.startswith('where'): buckets['where'].append(q)
    elif t.startswith(('is ','are ','does ','do ')): buckets['is_are'].append(q)
    else: buckets['other'].append(q)

per = max(1, NUM_SAMPLES // len(buckets))
chosen = []
for k, v in buckets.items():
    random.shuffle(v); chosen.extend(v[:per])
random.shuffle(chosen); chosen = chosen[:NUM_SAMPLES]
print('chosen', len(chosen))

COCO_TPL = 'http://images.cocodataset.org/val2014/COCO_val2014_{:012d}.jpg'
def fetch_image(image_id):
    fname = f'COCO_val2014_{image_id:012d}.jpg'
    p = IMG_DIR / fname
    if not p.exists():
        urllib.request.urlretrieve(COCO_TPL.format(image_id), p)
    return Image.open(p).convert('RGB')

samples = []
for q in tqdm(chosen):
    try:
        img = fetch_image(q['image_id'])
    except Exception as e:
        print('skip', q['image_id'], e); continue
    samples.append({
        'image': img,
        'question': q['question'],
        'ground_truth': annos[q['question_id']]['multiple_choice_answer'],
        'sample_id': str(q['question_id']),
        'image_id':  str(q['image_id']),
    })
print('ready', len(samples))

In [ ]:
TOP_VQA_ANSWERS = ['yes','no','2','1','white','3','red','blue','4','green','black','yellow','brown','right','left','man','woman','wood','table','blue and white','5','stop','6','gray','tennis','baseball','skateboarding','surfing','kite','snow','grass','water','tree','trees','food','pizza','cat','dog','bird','0','standing','sitting','walking','eating','sleeping','cake','laptop','phone','clock','car','truck','bus','train','bike','horse','sheep','cow','elephant','giraffe','zebra','bear','kitchen','living room','bedroom','bathroom','beach','park','street','ocean','sky','sunny','cloudy','day','night','summer','winter','plastic','metal','glass','paper','ceramic','leather','cotton','old','young','big','small','tall','short','happy','sad','frisbee','ball','racket','bat','glove','book','computer','tv','sandwich','apple','orange','banana','carrot','broccoli','donut','hot dog','wine','coffee','tea','water','milk']

def is_correct(pred, sample):
    pred = pred.strip().lower()
    if not pred: return False
    if pred == sample['ground_truth'].lower(): return True
    alts = [a['answer'].lower() for a in annos[int(sample['sample_id'])].get('answers', [])]
    return pred in alts

In [ ]:
import open_clip
print('loading CLIP ViT-B/32')
clip_model, _, clip_pp = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_tok = open_clip.get_tokenizer('ViT-B-32')
clip_model = clip_model.to(device, dtype=DTYPE).eval()

cap = {}
last_block = clip_model.visual.transformer.resblocks[-1]
_orig = last_block.attn.forward
@functools.wraps(_orig)
def _patched(q, k, v, **kw):
    kw['need_weights'] = True; kw['average_attn_weights'] = False
    out, w = _orig(q, k, v, **kw)
    cap['weights'] = w.detach()
    return out, w
last_block.attn.forward = _patched

with torch.no_grad():
    cand = clip_tok(TOP_VQA_ANSWERS).to(device)
    cand_feat = clip_model.encode_text(cand)
    cand_feat = cand_feat / cand_feat.norm(dim=-1, keepdim=True)
    cand_feat = cand_feat.to(DTYPE)

clip_results = []
for s in tqdm(samples, desc='CLIP'):
    cap.clear()
    img_t = clip_pp(s['image']).unsqueeze(0).to(device, dtype=DTYPE)
    with torch.no_grad():
        f = clip_model.encode_image(img_t)
        f = f / f.norm(dim=-1, keepdim=True)
        sims = (f.float() @ cand_feat.float().T).squeeze(0)
        idx = sims.argmax().item()
        pred = TOP_VQA_ANSWERS[idx]
    attn = cap.get('weights')
    if attn is not None:
        a = attn[0].float().mean(0)
        cls = a[0, 1:50].cpu().numpy()
        grid = cls.reshape(7, 7)
        grid = (grid - grid.min()) / (grid.max() - grid.min() + 1e-8)
        grid14 = np.kron(grid, np.ones((2, 2), dtype=np.float32))
        weights = grid14.tolist()
    else:
        weights = [[0.0]*14 for _ in range(14)]
    clip_results.append({
        'sample_id':s['sample_id'], 'image_id':s['image_id'], 'model':'CLIP',
        'question':s['question'], 'ground_truth':s['ground_truth'],
        'prediction':pred, 'correct':is_correct(pred, s),
        'confidence':float(sims.softmax(-1).max().item()),
        'weights':weights,
    })
del clip_model, clip_pp, clip_tok, cand_feat
gc.collect(); torch.cuda.empty_cache()
print('CLIP acc', sum(r['correct'] for r in clip_results), '/', len(clip_results))

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration

def run_blip2(model_id, tag):
    print('loading', model_id)
    proc = Blip2Processor.from_pretrained(model_id)
    m = Blip2ForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map='auto').eval()
    m.qformer.config.output_attentions = True
    cap = {}
    def hook(_mod, _i, output):
        # Blip2QFormerMultiHeadAttention.forward returns (context, attn_probs)
        if isinstance(output, tuple):
            for x in output:
                if torch.is_tensor(x) and x.dim() == 4:
                    cap['weights'] = x.detach(); return
    # 兼容老/新 transformers：老版本是 .attention.self，新版本是 .attention 直接
    xattn = m.qformer.encoder.layer[-1].crossattention.attention
    target = getattr(xattn, 'self', xattn)
    target.register_forward_hook(hook)
    out_results = []
    for s in tqdm(samples, desc=tag):
        cap.clear()
        prompt = f"Question: {s['question']} Answer:"
        inputs = proc(images=s['image'], text=prompt, return_tensors='pt').to(device, DTYPE)
        with torch.no_grad():
            o = m.generate(**inputs, max_new_tokens=8, do_sample=False)
        text = proc.tokenizer.decode(o[0], skip_special_tokens=True)
        pred = text.replace(prompt, '').strip()
        if '\n' in pred: pred = pred.split('\n')[0].strip()
        attn = cap.get('weights')
        if attn is not None:
            a = attn[0].float().mean(0).cpu().numpy()      # (32, 257)
            patch = a[:, 1:257]
            heat = patch.mean(0).reshape(16, 16)[1:15, 1:15]
            heat = (heat - heat.min())/(heat.max() - heat.min() + 1e-8)
            weights = heat.tolist()
        else:
            weights = [[0.0]*14 for _ in range(14)]
        out_results.append({
            'sample_id':s['sample_id'], 'image_id':s['image_id'], 'model':tag,
            'question':s['question'], 'ground_truth':s['ground_truth'],
            'prediction':pred, 'correct':is_correct(pred, s),
            'confidence':0.5, 'weights':weights,
        })
    del m, proc
    gc.collect(); torch.cuda.empty_cache()
    print(tag, 'acc', sum(r['correct'] for r in out_results), '/', len(out_results))
    return out_results

blip2_results    = run_blip2('Salesforce/blip2-opt-2.7b',   'BLIP2')
blip2_t5_results = run_blip2('Salesforce/blip2-flan-t5-xl', 'BLIP2-T5')

In [ ]:
# 视图五数据：纯文本基线（OPT-2.7b 解码器，不喂图像）
from transformers import Blip2Processor, Blip2ForConditionalGeneration
MID = 'Salesforce/blip2-opt-2.7b'
proc = Blip2Processor.from_pretrained(MID)
m = Blip2ForConditionalGeneration.from_pretrained(MID, torch_dtype=DTYPE, device_map='auto').eval()
blank = Image.new('RGB', (224, 224), (127, 127, 127))
text_bias_results = []
for s in tqdm(samples, desc='text-only'):
    prompt = f"Question: {s['question']} Answer:"
    inputs = proc(images=blank, text=prompt, return_tensors='pt').to(device, DTYPE)
    with torch.no_grad():
        o = m.generate(**inputs, max_new_tokens=8, do_sample=False)
    text = proc.tokenizer.decode(o[0], skip_special_tokens=True)
    pred = text.replace(prompt, '').strip()
    if '\n' in pred: pred = pred.split('\n')[0].strip()
    text_bias_results.append({
        'sample_id': s['sample_id'],
        'prediction': pred,
        'correct': is_correct(pred, s),
    })
del m, proc
gc.collect(); torch.cuda.empty_cache()
print('text-only acc', sum(r['correct'] for r in text_bias_results), '/', len(text_bias_results))

In [ ]:
all_results = clip_results + blip2_results + blip2_t5_results
tb_idx = {r['sample_id']: r for r in text_bias_results}
blip2_idx = {r['sample_id']: r for r in blip2_results}

out = {'matrix_data': [], 'attention_data': {}, 'text_bias': []}
for r in all_results:
    out['matrix_data'].append({
        'model': r['model'], 'sample_id': r['sample_id'],
        'correct': bool(r['correct']), 'confidence': float(r['confidence']),
    })
    img_url = f"http://images.cocodataset.org/val2014/COCO_val2014_{int(r['image_id']):012d}.jpg"
    out['attention_data'][f"{r['model']}__{r['sample_id']}"] = {
        'image_url': img_url, 'question': r['question'],
        'ground_truth': r['ground_truth'], 'prediction': r['prediction'],
        'weights': r['weights'],
    }
for sid, tb in tb_idx.items():
    if sid not in blip2_idx: continue
    out['text_bias'].append({
        'sample_id': sid,
        'question': blip2_idx[sid]['question'],
        'ground_truth': blip2_idx[sid]['ground_truth'],
        'text_only_pred': tb['prediction'],
        'text_only_correct': bool(tb['correct']),
        'with_image_pred': blip2_idx[sid]['prediction'],
        'with_image_correct': bool(blip2_idx[sid]['correct']),
    })
with open('vl_attention_data.json', 'w') as f:
    json.dump(out, f, indent=2)
print('saved vl_attention_data.json',
      'matrix', len(out['matrix_data']),
      'attn', len(out['attention_data']),
      'text_bias', len(out['text_bias']))

In [ ]:
# 一键拷贝到前端目录
import shutil
shutil.copy('vl_attention_data.json', '../src/data/vl_attention_data.json')
print('copied to ../src/data/vl_attention_data.json')